# 木探索中の確率事象を固定する

木探索は各分岐で実際に技を発動させて評価するため、命中判定・ダメージ乱数などの
確率的要素がそのまま評価値に混ざり込む。命中率80%の技を含む盤面で
evaluate_commands() を2回呼んでも、シミュレーションのたびに命中/外れが変わり得るため
評価値が一致するとは限らない。configure_sim() の役割・引数の詳細は
docs/api/README.md の TreeSearchPlayer「オーバーライド可能なフック」節を参照。

In [14]:
%pip install -q jpoke

Note: you may need to restart the kernel to use updated packages.


In [15]:
from jpoke import Battle
from jpoke.players import RandomPlayer, TreeSearchPlayer
from jpoke.enums import Command

In [16]:
class DeterministicSearchPlayer(TreeSearchPlayer):
    """探索中だけ命中率100%・ダメージを平均値に固定するAI（configure_sim()の拡張例）。"""

    def configure_sim(self, sim: Battle) -> None:
        sim.option.accuracy_fix_threshold = 70 # 命中率0%以上の技（=すべての技）を100%命中扱いにする
        sim.option.damage_roll = "average" # ダメージ乱数を毎回同じ平均値に固定する

    def choose_command(self, battle: Battle) -> Command:
        # evaluate_commands()は副作用なしのメソッド（04_command_evaluation_debug.ipynbの
        # DebugPlayerと同様）なので、2回呼んで表示を挟んでも探索本体の判断には影響しない。
        # configure_sim()で確率的要素を固定しているため、2回の評価値は毎回一致するはず
        table1 = self.evaluate_commands(battle)
        table2 = self.evaluate_commands(battle)
        if table1:
            print("1回目の評価値:", {str(c): round(v, 2) for c, v in table1.items()})
            print("2回目の評価値:", {str(c): round(v, 2) for c, v in table2.items()})
            print(f"2回の評価値が完全に一致: {table1 == table2}")
        return super().choose_command(battle)

In [17]:
# ハイドロポンプは命中率80%。命中判定が絡む技を含む盤面で比較する
ai_player = DeterministicSearchPlayer("DeterministicAI", max_plies=1, max_nodes=50)
ai_player.add_pokemon("カメックス", moves=["ハイドロポンプ", "たいあたり"])

opponent = RandomPlayer("Opponent")
opponent.add_pokemon("カビゴン", moves=["のしかかり"])

battle = Battle(ai_player, opponent)
battle.start()
battle.step()
battle.step()

1回目の評価値: {'MOVE_0': -0.07, 'MOVE_1': -0.3}
2回目の評価値: {'MOVE_0': -0.07, 'MOVE_1': -0.3}
2回の評価値が完全に一致: True
